<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9E%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_100_ml_ozon_recsys_baseline_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
# НОВЫЙ ПУТЬ: Добавляем путь к данным взаимодействий за нужный период
TRACKER_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_tracker_data/final_apparel_tracker_data_08'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    print("Загружаем тренировочные данные заказов...")
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if 'created_date' in df.columns and df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    elif 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        # Если created_date вообще отсутствует, создаем её
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])
    print(f"Загружено заказов: {len(df):,}")
    return df

orders_df = load_orders()

Mounted at /content/drive
Загружаем тренировочные данные заказов...
Загружено заказов: 20,362,338


In [2]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [3]:
# Шаг 2.5. Загрузка взаимодействий (tracker)
def load_tracker():
    print("Загружаем тренировочные данные взаимодействий...")
    tracker_data = []
    # Используем rglob для рекурсивного поиска всех .parquet файлов
    # в директории TRACKER_PATH и её поддиректориях
    for f in Path(TRACKER_PATH).rglob('*.parquet'):
        tracker_data.append(pd.read_parquet(f))

    if tracker_data:
        df = pd.concat(tracker_data, ignore_index=True)
        print(f"Загружено взаимодействий: {len(df):,}")
        return df
    else:
        print("Файлы взаимодействий не найдены.")
        return pd.DataFrame() # Возвращаем пустой DataFrame

tracker_df = load_tracker()

Загружаем тренировочные данные взаимодействий...
Загружено взаимодействий: 116,552,409


In [4]:
print("\n\n=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(tracker_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in tracker_df.columns:
    print(f"  • {col}: {tracker_df[col].dtype}")



=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------------+-----------+-----------+---------------------+---------------+
|    | action_widget   |   item_id |   user_id | timestamp           | action_type   |
+====+=================+===========+===========+=====================+===============+
|  0 | pdp             |    996252 |   3542470 | 2025-07-08 23:29:23 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+
|  1 | pdp             |   4127285 |   3267450 | 2025-07-09 14:51:02 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+

📊 Схема данных:
  • action_widget: object
  • item_id: int32
  • user_id: int32
  • timestamp: datetime64[ns]
  • action_type: object


In [5]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]

Найдено уникальных тестовых пользователей: 470,347


In [6]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [7]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 20,362,338
Уникальных пользователей: 842,254
Уникальных товаров: 4,679,218
Период данных: 2025-01-01 - 2025-07-15

Распределение статусов заказов:
  delivered_orders: 10,420,894 (51.2%)
  canceled_orders: 8,420,631 (41.4%)
  proccesed_orders: 1,520,813 (7.5%)


In [8]:
print("\n\nАНАЛИЗ ВЗАИМОДЕЙСТВИЙ")
print("=" * 50)
print(f"Общее количество взаимодействий: {len(tracker_df):,}")
print(f"Уникальных пользователей: {tracker_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {tracker_df['item_id'].nunique():,}")

if 'timestamp' in tracker_df.columns:
    min_date = tracker_df['timestamp'].min().date()
    max_date = tracker_df['timestamp'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'action_type' in tracker_df.columns:
    print("\nРаспределение типов действий:")
    action_counts = tracker_df['action_type'].value_counts()
    action_counts_pct = tracker_df['action_type'].value_counts(normalize=True) * 100
    for action, count in action_counts.items():
        pct = action_counts_pct[action]
        print(f"  {action}: {count:,} ({pct:.1f}%)")



АНАЛИЗ ВЗАИМОДЕЙСТВИЙ
Общее количество взаимодействий: 116,552,409
Уникальных пользователей: 807,670
Уникальных товаров: 3,197,923
Период данных: 2010-01-30 - 2025-07-16

Распределение типов действий:
  page_view: 80,605,340 (69.2%)
  view_description: 17,253,961 (14.8%)
  review_view: 5,729,054 (4.9%)
  to_cart: 4,618,070 (4.0%)
  favorite: 3,455,017 (3.0%)
  remove: 3,007,598 (2.6%)
  unfavorite: 1,883,369 (1.6%)


In [16]:
# Для tracker_df по timestamp
if 'timestamp' in tracker_df.columns:
    print("\nКоличество взаимодействий по годам (на основе timestamp):")
    tracker_by_year = tracker_df['timestamp'].dt.year.value_counts().sort_index()
    for year, count in tracker_by_year.items():
        print(f"  {year}: {count:,}")


Количество взаимодействий по годам (на основе timestamp):
  2010: 1
  2024: 12
  2025: 116,552,396


In [17]:
print("\n\nАНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)")
print("=" * 50)
# Объединим item_id из orders и tracker для более полной картины
all_item_ids = set(orders_df['item_id'].unique()).union(set(tracker_df['item_id'].unique()))
print(f"Общее количество уникальных товаров в заказах и взаимодействиях: {len(all_item_ids):,}")



АНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)
Общее количество уникальных товаров в заказах и взаимодействиях: 4,797,217


In [10]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 51974017: 13,361 заказов
  Товар 187052809: 12,384 заказов
  Товар 207631139: 8,877 заказов
  Товар 143497612: 4,096 заказов
  Товар 119105606: 3,497 заказов
  Товар 247423473: 3,188 заказов
  Товар 77696741: 2,735 заказов
  Товар 175287070: 2,725 заказов
  Товар 285009143: 2,634 заказов
  Товар 201930716: 2,624 заказов
